# 73. 초기 학습률을 0.0001로 낮추면 원판 궤적이 더 정확해질까?

같은 선형끼리, 같은 코사인끼리 0.0005 → 0.0001을 비교합니다. 두 새 모델은 같은 RTX 3090에서 원본 YOLO26n으로 각각 시작했습니다. 최대 200에폭·patience 30·시드 0과 기존 선수별 분할을 유지했습니다. 아래 셀들은 Mac에 저장한 결과를 읽습니다.

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from wlcv.config import ROOT

previous = ROOT / "outputs/plate_lr_grid/20261126"
experiment = ROOT / "outputs/plate_lr_0001/20261126"
review = experiment / "review"
runs = {
    "lr0005_linear": previous, "lr0005_cosine": previous,
    "lr0001_linear": experiment, "lr0001_cosine": experiment,
}
history = {}
audit = {}
for name, folder in runs.items():
    run = folder / "runs" / name
    history[name] = pd.read_csv(run / "results.csv").set_index("epoch")
    audit[name] = json.loads((run / "audit.json").read_text())
display(pd.DataFrame(audit).T[["initial_lr", "cos_lr", "best_epoch", "epochs_run", "train_and_val_seconds"]])

## 실제 학습률과 검증 성능

실선은 선형, 점선은 코사인입니다. 검증 곡선은 최근 5에폭 평균이며 학습이 멈춘 지점에서 끝납니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7), layout="constrained")
for ax, (column, title) in zip(axes, [("lr/pg0","Recorded learning rate"), ("val/box_loss","Validation box loss"), ("metrics/mAP50-95(B)","Validation mAP50-95")]):
    for name, log in history.items():
        data = log[column] if column == "lr/pg0" else log[column].rolling(5, min_periods=1).mean()
        lr = audit[name]["initial_lr"]
        schedule = "cosine" if audit[name]["cos_lr"] else "linear"
        ax.plot(data, label=f"{lr} {schedule}", color="#1c6994" if lr == 0.0005 else "#b14524",
                linestyle="--" if schedule == "cosine" else "-")
    ax.set(title=title, xlabel="Epoch")
    ax.grid(alpha=.2)
axes[0].legend(fontsize=8)
fig.savefig(review / "training_curves.png", dpi=145)
plt.show()

## 같은 검증셋에서 무엇이 달라졌나

동작 중 수동 박스 270장에서 미탐·오탐을 세고 동작 전체 478장에서 좌표 누락을 셉니다. 상승량은 무릎 통과→접촉의 원판 중심 y좌표 차이입니다. 선수별 오차가 양수이면 수동 상승량보다 크게 측정했습니다. 평균은 모든 모델에서 비교할 수 있는 같은 선수를 사용합니다.

In [ ]:
summary = pd.read_csv(review / "summary.csv").set_index("case")
fp = pd.read_csv(review / "false_positives.csv")
motion_fp = fp[fp.phase == "동작 중"].set_index("case").fp_frames
display(summary.assign(motion_fp=motion_fp).round(4))
diff = pd.read_csv(review / "contrasts.csv")
display(diff[diff.after.str.startswith("lr0001")].round(4))
people = pd.read_csv(review / "athletes.csv")
display(people.pivot(index="video_id", columns="case", values="rise_error_px").reindex(columns=summary.index).round(2))

## a03에서 언제 궤적이 끊겼나

색이 있는 칸은 시간 연결 후에도 원판 중심이 없는 프레임입니다. 같은 프레임 번호를 세로로 비교하세요.

In [ ]:
from matplotlib.colors import ListedColormap
frames = pd.read_csv(review / "frames.csv")
a03 = frames[(frames.video_id == "a03_sn_f_01") & frames.in_motion].copy()
a03["missing"] = a03.cy.isna()
gaps = a03.pivot(index="case", columns="frame_index", values="missing").loc[list(runs)]
fig, ax = plt.subplots(figsize=(11, 2.6), layout="constrained")
ax.imshow(gaps.to_numpy(dtype=int), aspect="auto", interpolation="nearest", cmap=ListedColormap(["#e0e7e9","#9a3412"]),
          vmin=0, vmax=1, extent=[gaps.columns.min()-.5, gaps.columns.max()+.5, len(gaps)-.5, -.5])
ax.set(yticks=range(len(gaps)), yticklabels=gaps.index, xlabel="Frame", title="a03 | Missing plate center after temporal linking")
fig.savefig(review / "a03_gaps.png", dpi=145)
plt.show()

## 대표 프레임 검수

b02의 바와 몸이 접촉하는 36번에서 원판 바깥 박스와 허브를 봅니다. 초록 점선은 수동 박스, 주황 실선은 선택된 박스, 흰색 십자는 직접 찍은 허브입니다.

In [ ]:
from matplotlib.patches import Rectangle
from wlcv.config import ANNOTATIONS, VIDEOS
from wlcv.data.frames import read_rgb

video, index = "b02_sn_h_02", 36
frames = pd.read_csv(review / "frames.csv")
rows = frames[(frames.video_id == video) & (frames.frame_index == index)].set_index("case")
hubs = pd.read_csv(ANNOTATIONS / "plate_blur_review/validation.csv")
hub = hubs[(hubs.video_id == video) & (hubs.frame_index == index)].iloc[0]
rgb = read_rgb(VIDEOS / f"{video}.mp4", index)
fig, axes = plt.subplots(1, len(runs), figsize=(12, 3.4), layout="constrained")
for ax, name in zip(axes, runs):
    row = rows.loc[name]
    ax.imshow(rgb)
    x, y = row.gt_x1, row.gt_y1
    w, h = row.gt_x2 - x, row.gt_y2 - y
    ax.add_patch(Rectangle((x,y), w, h, fill=False, edgecolor="lime", linestyle="--", linewidth=2))
    if pd.notna(row.x1):
        ax.add_patch(Rectangle((row.x1,row.y1),row.x2-row.x1,row.y2-row.y1,
                               fill=False,edgecolor="darkorange",linewidth=2))
    ax.plot(hub.manual_x,hub.manual_y,"w+",markersize=10,markeredgewidth=2)
    ax.set(xlim=(max(0,x-.3*w),min(rgb.shape[1],x+1.3*w)),
           ylim=(min(rgb.shape[0],y+1.3*h),max(0,y-.3*h)), title=f"{name}\nIoU {row.iou:.2f}")
    ax.axis("off")
fig.savefig(review / "b02_contact_local.png", dpi=140)
plt.show()